# Monte Carlo Simulation for TASCM vs SCM

- repetitions = 3
- CSV outputs -> simulation_outputs/data/
- PDF plots -> simulation_outputs/plots/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(42)
REPETITIONS = 3
J, T0, T1 = 10, 80, 20
T = T0 + T1

OUT_DIR = Path('simulation_outputs')
DATA_DIR = OUT_DIR / 'data'
PLOT_DIR = OUT_DIR / 'plots'
DATA_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def simplex_projection(v):
    v = np.asarray(v, dtype=float)
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho = np.nonzero(u * np.arange(1, len(v) + 1) > (cssv - 1))[0][-1]
    theta = (cssv[rho] - 1) / (rho + 1.0)
    return np.maximum(v - theta, 0)

def ar1_series(T, phi=0.7, sigma=1.0):
    x = np.zeros(T)
    eps = np.random.normal(0, sigma, size=T)
    for t in range(1, T):
        x[t] = phi * x[t-1] + eps[t]
    return x

def generate_panel(J=10, T=100, T0=80):
    common = ar1_series(T, phi=0.6, sigma=1.0)
    donors = []
    for _ in range(J):
        donor = 0.8 * common + 0.4 * ar1_series(T, phi=np.random.uniform(0.3, 0.9), sigma=0.8)
        donors.append(donor)
    donors = np.vstack(donors)
    true_w = np.random.dirichlet(np.ones(J))
    cyc = 0.5 * np.sin(2 * np.pi * np.arange(T) / 12 + np.random.uniform(0, 2*np.pi))
    treated0 = true_w @ donors + cyc + np.random.normal(0, 0.2, size=T)
    tau = np.zeros(T); tau[T0:] = 1.5
    treated_obs = treated0 + tau
    return donors, treated0, treated_obs, tau

def fit_scm(y_pre, X_pre, n_iter=500, lr=0.05):
    J = X_pre.shape[0]
    w = np.ones(J) / J
    for _ in range(n_iter):
        grad = 2 * X_pre @ (X_pre.T @ w - y_pre) / len(y_pre)
        w = simplex_projection(w - lr * grad)
    return w

def topo_feature(x):
    x = (x - x.mean()) / (x.std() + 1e-8)
    ac1 = np.corrcoef(x[:-1], x[1:])[0,1] if len(x) > 2 else 0.0
    spec = np.abs(np.fft.rfft(x))
    p = spec / (spec.sum() + 1e-12)
    ent = -(p * np.log(p + 1e-12)).sum()
    return np.array([ac1, ent])

def fit_tascm(y_pre, X_pre, lam=0.5, n_iter=300, lr=0.03):
    J, T0_local = X_pre.shape
    w = np.ones(J) / J
    target = topo_feature(y_pre)
    for _ in range(n_iter):
        yhat = X_pre.T @ w
        grad_y = 2 * X_pre @ (yhat - y_pre) / T0_local
        eps = 1e-4
        grad_top = np.zeros(J)
        base = np.sum((topo_feature(yhat) - target) ** 2)
        for j in range(J):
            w2 = w.copy(); w2[j] += eps; w2 = simplex_projection(w2)
            yhat2 = X_pre.T @ w2
            loss2 = np.sum((topo_feature(yhat2) - target) ** 2)
            grad_top[j] = (loss2 - base) / eps
        w = simplex_projection(w - lr * (grad_y + lam * grad_top))
    return w

def rmspe(y, yhat):
    return np.sqrt(np.mean((y - yhat) ** 2))

In [ ]:
rows, path_rows = [], []
for r in range(1, REPETITIONS + 1):
    donors, treated0, treated_obs, tau = generate_panel(J=J, T=T, T0=T0)
    y_pre, X_pre = treated_obs[:T0], donors[:, :T0]
    w_scm = fit_scm(y_pre, X_pre)
    w_ta = fit_tascm(y_pre, X_pre, lam=0.5)
    scm_cf = donors.T @ w_scm
    ta_cf = donors.T @ w_ta

    rows.append({
        'rep': r,
        'pre_rmspe_scm': rmspe(treated0[:T0], scm_cf[:T0]),
        'post_rmspe_scm': rmspe(treated0[T0:], scm_cf[T0:]),
        'pre_rmspe_tascm': rmspe(treated0[:T0], ta_cf[:T0]),
        'post_rmspe_tascm': rmspe(treated0[T0:], ta_cf[T0:])
    })

    df_rep = pd.DataFrame({
        't': np.arange(T),
        'treated_obs': treated_obs,
        'treated_0_true': treated0,
        'scm_cf': scm_cf,
        'tascm_cf': ta_cf,
        'tau_true': tau,
        'rep': r
    })
    df_rep.to_csv(DATA_DIR / f'rep_{r:02d}_paths.csv', index=False)
    path_rows.append(df_rep)

results = pd.DataFrame(rows)
results.to_csv(DATA_DIR / 'montecarlo_summary.csv', index=False)
results

In [ ]:
plot_df = results.melt(id_vars='rep', value_vars=['post_rmspe_scm', 'post_rmspe_tascm'],
                       var_name='method', value_name='post_rmspe')
plt.figure(figsize=(7,4))
for m, g in plot_df.groupby('method'):
    plt.plot(g['rep'], g['post_rmspe'], marker='o', label=m)
plt.xlabel('Repetition')
plt.ylabel('Post-treatment RMSPE')
plt.title('Monte Carlo (R=3): Post-treatment RMSPE')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(PLOT_DIR / 'post_rmspe_by_rep.pdf')
plt.show()

In [ ]:
all_paths = pd.concat(path_rows, ignore_index=True)
avg = all_paths.groupby('t', as_index=False)[['treated_0_true','scm_cf','tascm_cf']].mean()
plt.figure(figsize=(8,4))
plt.plot(avg['t'], avg['treated_0_true'], label='True untreated', linewidth=2)
plt.plot(avg['t'], avg['scm_cf'], label='SCM', linestyle='--')
plt.plot(avg['t'], avg['tascm_cf'], label='TASCM', linestyle='-.')
plt.axvline(T0-1, color='k', alpha=0.4, linestyle=':')
plt.title('Average Counterfactual Paths')
plt.xlabel('Time'); plt.ylabel('Outcome')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(PLOT_DIR / 'average_counterfactual_paths.pdf')
plt.show()